# Overview

[Imagen en Vertex AI](https://cloud.google.com/vertex-ai/docs/generative-ai/image/overview) lleva las capacidades de IA generativa de vanguardia de Google a los desarrolladores de aplicaciones. Con Imagen en Vertex AI, los desarrolladores de aplicaciones pueden crear productos de IA de próxima generación que transforman la imaginación de sus usuarios en activos visuales de alta calidad, en segundos.

Con Imagen, puedes hacer lo siguiente:
- Generar imágenes nuevas a partir de un prompt.
- Editar una imagen completa cargada o generada con una indicación de texto.
- Editar solo partes de una imagen cargada o generada usando un área de máscara que definas.
- Aumentar la resolución de imágenes existentes, generadas o editadas.
- Ajustar un modelo con un tema específico (por ejemplo, un bolso o zapato específico) para la generación de imágenes.
- Obtener descripciones de texto de imágenes con subtítulos visuales.
- Obtener respuestas a una pregunta sobre una imagen con Visual Question Answering (VQA).

# Objetivos
En este cuaderno, exploraremos algunas de las funciones de *edición de imágenes de Imagen* utilizando el SDK de Python de Vertex AI.

Vamos a realizar diversos ejercicios

## Edición de imágenes
1. `product-background` - Edición de backgrounds de imágenes existentes
2. `ouptaint` - Expansión de imágenes existentes a nuevos formatos

### Set-up: SDK de Python de Vertex AI y depencencias necesarias

In [ ]:
%pip install --upgrade --quiet google-genai

### Autentica tu entorno de cuaderno (solo Colab)

Si estás ejecutando este cuaderno en Google Colab, ejecuta la siguiente celda para autenticar tu entorno.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Importa las librerias necesarias

In [ ]:
from google import genai
from google.genai.types import (
    EditImageConfig,
    GenerateImagesConfig,
    Image,
    MaskReferenceConfig,
    MaskReferenceImage,
    RawReferenceImage,
)

### Inicializar Vertex AI en nuestro Google Cloud Project

In [ ]:
# Importar Vertex AI
import vertexai

# Definir la info del proyecto
PROJECT_ID = ""  # @param {type:"string"}
# Vamos a utilizar esta localización por defecto
LOCATION = "us-central1"

# Inicializar el módulo
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

Vamos a verificar que todo está en orden y qué modo estamos utilizando

In [ ]:
if not client.vertexai:
    print("Usando Gemini Developer API.")
elif client._api_client.project:
    print(
        f"Usando Vertex AI en el proyecto: {client._api_client.project} en la localización: {client._api_client.location}"
    )
elif client._api_client.api_key:
    print(
        f"Usando Vertex AI en modo express con API key: {client._api_client.api_key[:5]}...{client._api_client.api_key[-5:]}"
    )

###  Funciones auxiliares

In [ ]:
import io
import urllib
import typing
import IPython.display
import math
import matplotlib.pyplot as plt

from PIL import ImageOps as PIL_ImageOps
from PIL import Image as PIL_Image
import matplotlib.pyplot as plt


# Gets the image bytes from a PIL Image object.
def get_bytes_from_pil(image: PIL_Image) -> bytes:
    byte_io_png = io.BytesIO()
    image.save(byte_io_png, "PNG")
    return byte_io_png.getvalue()


# Pads an image for outpainting.
def pad_to_target_size(
    source_image,
    target_size=(1536, 1536),
    mode="RGB",
    vertical_offset_ratio=0,
    horizontal_offset_ratio=0,
    fill_val=255,
):
    orig_image_size_w, orig_image_size_h = source_image.size
    target_size_w, target_size_h = target_size

    insert_pt_x = (target_size_w - orig_image_size_w) // 2 + int(
        horizontal_offset_ratio * target_size_w
    )
    insert_pt_y = (target_size_h - orig_image_size_h) // 2 + int(
        vertical_offset_ratio * target_size_h
    )
    insert_pt_x = min(insert_pt_x, target_size_w - orig_image_size_w)
    insert_pt_y = min(insert_pt_y, target_size_h - orig_image_size_h)

    if mode == "RGB":
        source_image_padded = PIL_Image.new(
            mode, target_size, color=(fill_val, fill_val, fill_val)
        )
    elif mode == "L":
        source_image_padded = PIL_Image.new(mode, target_size, color=(fill_val))
    else:
        raise ValueError("source image mode must be RGB or L.")

    source_image_padded.paste(source_image, (insert_pt_x, insert_pt_y))
    return source_image_padded


# Pads and resizes image and mask to the same target size.
def pad_image_and_mask(
    image_vertex: PIL_Image,
    mask_vertex: PIL_Image,
    target_size,
    vertical_offset_ratio,
    horizontal_offset_ratio,
):
    image_vertex.thumbnail(target_size)
    mask_vertex.thumbnail(target_size)

    image_vertex = pad_to_target_size(
        image_vertex,
        target_size=target_size,
        mode="RGB",
        vertical_offset_ratio=vertical_offset_ratio,
        horizontal_offset_ratio=horizontal_offset_ratio,
        fill_val=0,
    )
    mask_vertex = pad_to_target_size(
        mask_vertex,
        target_size=target_size,
        mode="L",
        vertical_offset_ratio=vertical_offset_ratio,
        horizontal_offset_ratio=horizontal_offset_ratio,
        fill_val=255,
    )
    return image_vertex, mask_vertex


def display_images(original_image, modified_image) -> None:
    fig, axis = plt.subplots(1, 2, figsize=(12, 6))
    axis[0].imshow(original_image)
    axis[0].set_title("Imagen original")
    axis[1].imshow(modified_image)
    axis[1].set_title("Imagen editada")
    for ax in axis:
        ax.axis("off")
    plt.show()


def display_image(
    image,
    max_width: int = 700,
    max_height: int = 400,
) -> None:
    pil_image = typing.cast(PIL_Image.Image, image._pil_image)
    if pil_image.mode != "RGB":
        # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
        pil_image = pil_image.convert("RGB")
    image_width, image_height = pil_image.size
    if max_width < image_width or max_height < image_height:
        # Resize to display a smaller notebook image
        pil_image = pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
    IPython.display.display(pil_image)

### Cargar los modelos de generación y edición de imágenes

  - Generación de imágenes - `imagen-4.0-fast-generate-001`
  - Edición de imágenes - `imagen-3.0-capability-001`

In [ ]:
generation_model = "imagen-3.0-generate-002"

edit_model = "imagen-3.0-capability-001"

### Ejercicio 1 - Inpainting insert

En estos ejemplos, especificarás un área objetivo para aplicar ediciones. En el caso de "inpainting insert", utilizarás un área de máscara para añadir contenido a una imagen existente.

1. Comienza generando una imagen con Imagen 3.
2. Luego, crea dos objetos
    - ```ReferenceImage```: uno para tu imagen de referencia y otro para tu máscara. Para el objeto
    - ```MaskReferenceImage```, establece ```reference_image=None```; esto permitirá la detección automática de la máscara basada en el ```mask_mode``` especificado.

In [ ]:
image_prompt = """
Un pequeño cuenco de madera con uvas y manzanas sobre una encimera de mármol en la cocina, con armarios de color marrón claro desenfocados al fondo
"""
generated_image_response = client.models.generate_images(
    model=generation_model,
    prompt=image_prompt,
    config=GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="1:1",
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="DONT_ALLOW",
    ),
)

display_image(generated_image_response.generated_images[0].image)

In [ ]:
edit_prompt = ""  # TODO: Completa el prompt para añadir contenido a la imagen
raw_ref_image = RawReferenceImage(
    reference_image=generated_image_response.generated_images[0].image, reference_id=0
)
mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=None,
    config=MaskReferenceConfig(
        mask_mode="",  # TODO: Completa el mask_mode correspondiente a FOREGROUND
        mask_dilation=0.1,
    ),
)
edited_image_response = client.models.edit_image(
    model="",  # TODO: Añade el modelo de generación de imágenes
    prompt="",  # TODO: Añade el prompt
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_images(
    generated_image_response.generated_images[0].image._pil_image,
    edited_image_response.generated_images[0].image._pil_image,
)

### Ejercicio 2 - Inpainting insert semántico

Este siguiente ejemplo demuestra otro caso de "inpainting insert" (inserción por repintado). Sin embargo, en esta ocasión utilizarás el modo de máscara semántica (`semantic mask mode`).

Al usar este modo, deberás especificar el ID de clase (`class ID`) del objeto en la imagen que deseas enmascarar y reemplazar.

### Clases de segmentación semántica

| ID de Clase | Tipo de Instancia | ID de Clase | Tipo de Instancia | ID de Clase | Tipo de Instancia | ID de Clase | Tipo de Instancia |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | mochila | 50 | zanahoria | 100 | acera_pavimento | 150 | esquís |
| 1 | paraguas | 51 | hot_dog | 101 | pista_aterrizaje | 151 | snowboard |
| 2 | bolso/bolsa | 52 | pizza | 102 | terreno | 152 | pelota_deportiva |
| 3 | corbata | 53 | dónut | 103 | libro | 153 | cometa |
| 4 | maleta | 54 | pastel | 104 | caja | 154 | bate_de_béisbol |
| 5 | estuche/funda | 55 | fruta_otra | 105 | reloj | 155 | guante_de_béisbol |
| 6 | pájaro | 56 | comida_otra | 106 | jarrón | 156 | monopatín |
| 7 | gato | 57 | silla_otra | 107 | tijeras | 157 | tabla_de_surf |
| 8 | perro | 58 | sillón | 108 | juguete_otro | 158 | raqueta_de_tenis |
| 9 | caballo | 59 | silla_giratoria | 109 | oso_de_peluche | 159 | red |
| 10 | oveja | 60 | taburete | 110 | secador_de_pelo | 160 | base |
| 11 | vaca | 61 | asiento | 111 | cepillo_de_dientes | 161 | escultura |
| 12 | elefante | 62 | sofá | 112 | pintura/cuadro | 162 | columna |
| 13 | oso | 63 | cubo_de_basura | 113 | póster | 163 | fuente |
| 14 | cebra | 64 | planta_en_maceta | 114 | tablón_anuncios | 164 | toldo |
| 15 | jirafa | 65 | mesita_de_noche | 115 | botella | 165 | vestimenta/ropa |
| 16 | animal_otro | 66 | cama | 116 | taza | 166 | pancarta |
| 17 | microondas | 67 | mesa | 117 | copa_de_vino | 167 | bandera |
| 18 | radiador | 68 | mesa_de_billar | 118 | cuchillo | 168 | manta |
| 19 | horno | 69 | barril | 119 | tenedor | 169 | cortina_otra |
| 20 | tostadora | 70 | escritorio | 120 | cuchara | 170 | cortina_de_ducha |
| 21 | tanque_almacén | 71 | otomana/puf | 121 | cuenco | 171 | almohada/cojín |
| 22 | cinta_transp. | 72 | armario/ropero | 122 | bandeja | 172 | toalla |
| 23 | fregadero | 73 | cuna | 123 | campana_extr. | 173 | alfombra/tapete |
| 24 | refrigerador | 74 | cesta | 124 | plato | 174 | vegetación |
| 25 | lav./secadora | 75 | cómoda | 125 | persona | 175 | bicicleta |
| 26 | ventilador | 76 | estantería | 126 | jinete_otro | 176 | coche |
| 27 | lavavajillas | 77 | encimera_otra | 127 | ciclista | 177 | motocarro |
| 28 | inodoro | 78 | encimera_baño | 128 | motociclista | 178 | motocicleta |
| 29 | bañera | 79 | isla_cocina | 129 | papel | 179 | avión |
| 30 | ducha | 80 | puerta | 130 | farola | 180 | autobús |
| 31 | túnel | 81 | luz_otra | 131 | barrera_vial | 181 | tren |
| 32 | puente | 82 | lámpara | 132 | buzón | 182 | camión |
| 33 | muelle | 83 | aplique_pared | 133 | cámara_cctv | 183 | remolque |
| 34 | tienda_campaña | 84 | araña_luces | 134 | caja_empalmes | 184 | barco/buque |
| 35 | edificio | 85 | espejo | 135 | señal_tráfico | 185 | obj_ruedas_lento |
| 36 | techo | 86 | pizarra_blanca | 136 | semáforo | 186 | río_lago |
| 37 | portátil | 87 | estante | 137 | hidrante | 187 | mar |
| 38 | teclado | 88 | escaleras | 138 | parquímetro | 188 | agua_otra |
| 39 | ratón | 89 | esc. mecánica | 139 | banco | 189 | piscina |
| 40 | mando_dist. | 90 | gabinete | 140 | portabicicletas | 190 | cascada |
| 41 | móvil | 91 | chimenea | 141 | valla_publi. | 191 | pared |
| 42 | televisión | 92 | estufa/fogón | 142 | cielo | 192 | ventana |
| 43 | suelo | 93 | máq. arcade | 143 | poste | 193 | persiana |
| 44 | escenario | 94 | grava | 144 | cerca/valla | | |
| 45 | plátano | 95 | plataforma | 145 | barandilla | | |
| 46 | manzana | 96 | campo_juego | 146 | guardarraíl | | |
| 47 | sándwich | 97 | vía_férrea | 147 | montaña | | |
| 48 | naranja | 98 | carretera | 148 | roca | | |
| 49 | brócoli | 99 | nieve | 149 | frisbee | | |

Una vez que hayas encontrado el ID de clase de segmentación correcto, inclúyelo en ```segmentation_classes```.

Dentro del objeto ```MaskReferenceImage```, también puedes configurar el valor de dilatación (`dilation`). Este valor decimal (float) entre 0 y 1 representa el porcentaje de expansión de la máscara proporcionada.

In [ ]:
image_prompt = """
Un bulldog francés sentado en la sala de estar sobre un sofá con cojines decorativos verdes y una manta, con un espejo circular en la pared encima del sofá
"""
generated_image_response = client.models.generate_images(
    model=generation_model,
    prompt=image_prompt,
    config=GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="1:1",
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="DONT_ALLOW",
    ),
)

display_image(generated_image_response.generated_images[0].image)

In [ ]:


edit_prompt = ""  # TODO: Completa el prompt para añadir contenido a la imagen
raw_ref_image = RawReferenceImage(
    reference_image=generated_image_response.generated_images[0].image, reference_id=0
)
mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=None,
    config=MaskReferenceConfig(
        mask_mode="",  # TODO: Completa el mask_mode correspondiente a SEMANTICO,
        segmentation_classes=[],  # TODO: Completa con los ids correspondientes a reemplazar
        mask_dilation=0.1,
    ),
)
edited_image_response = client.models.edit_image(
    model="",  # TODO: Añade el modelo de generación de imágenes
    prompt="",  # TODO: Añade el prompt
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_images(
    generated_image_response.generated_images[0].image._pil_image,
    edited_image_response.generated_images[0].image._pil_image,
)

### Ejercicio 3 - Inpainting remove

"Inpainting remove" permite utilizar un área de máscara para eliminar contenido de una imagen.

En este siguiente ejemplo, tomarás una imagen de Google Cloud Storage que muestra una pared con un espejo y algunas fotos, y crearás una máscara sobre las instancias del espejo detectadas. Luego, eliminarás este objeto configurando el modo de edición como "EDIT_MODE_INPAINT_REMOVAL". Para este tipo de solicitudes, el prompt puede ser una cadena de texto vacía.

In [ ]:

starting_image_show = PIL_Image.open(
    urllib.request.urlopen(
        "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/mirror.png"
    )
)
starting_image_show.thumbnail((400, 400))

display(starting_image_show)


In [ ]:

starting_image = Image(gcs_uri="gs://cloud-samples-data/generative-ai/image/mirror.png")
raw_ref_image = RawReferenceImage(reference_image=starting_image, reference_id=0)

mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=None,
    config=MaskReferenceConfig(
        mask_mode="",  # TODO: Completa el mask_mode correspondiente a SEMANTICO,
        segmentation_classes=[],  # TODO: Completa con los ids correspondientes a reemplazar
    ),
)

remove_image = client.models.edit_image(
    model="",  # TODO: Añade el modelo de generación de imágenes
    prompt="",  # TODO: Añade el prompt
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_images(
    starting_image_show,
    remove_image.generated_images[0].image._pil_image,
)

### Ejercicio 4 - Edición de fondo de producto mediante el modo "background swap"

También puedes utilizar Imagen 3 para la edición de imágenes de productos. Al configurar el `edit_mode` como "EDIT_MODE_BGSWAP", puedes mantener el contenido del producto mientras modificas el fondo de la imagen.

Para este ejemplo, comienza con una imagen almacenada en un cubo (bucket) de Google Cloud Storage y proporciona un prompt que describa la nueva escena del fondo.

In [ ]:
starting_image_show = PIL_Image.open(
    urllib.request.urlopen(
        "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png"
    )
)
starting_image_show.thumbnail((400, 400))

display(starting_image_show)

In [ ]:
product_image = Image(
    gcs_uri="gs://cloud-samples-data/generative-ai/image/suitcase.png"
)
raw_ref_image = RawReferenceImage(reference_image=product_image, reference_id=0)

mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=None,
    config=MaskReferenceConfig(
        mask_mode=""  # TODO: Completa el mask_mode correspondiente a BACKGROUND
    ),
)

prompt = "Una maleta de color azul claro frente a una ventana en un aeropuerto, con mucha luz natural y brillante entrando por los ventanales, y aviones despegando a lo lejos"
edited_image = client.models.edit_image(
    model="",  # TODO: Añade el modelo de generación de imágenes
    prompt="",  # TODO: Añade el prompt
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

product_image_show = PIL_Image.open(
    urllib.request.urlopen(
        "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png"
    )
)
display_images(product_image_show, edited_image.generated_images[0].image._pil_image)

### Ejercicio 5 - Outpainting - Extensión de una imágen a otro formato

La edición con Imagen 3 puede utilizarse para el "outpainting" de imágenes. El *outpainting* se utiliza para expandir el contenido de una imagen a un área más grande o a un área con dimensiones diferentes. Para utilizar la función de *outpainting*, debes crear una máscara de imagen y preparar la imagen original añadiendo un margen de espacio vacío (padding) a su alrededor. Una vez que hayas aplicado el relleno a la imagen, puedes usar el modo de edición ```outpainting``` para completar el espacio vacío.

In [ ]:
starting_image_show = PIL_Image.open(
    urllib.request.urlopen(
        "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/living-room.png"
    )
)
starting_image_show.thumbnail((400, 400))

display(starting_image_show)

In [ ]:
! gcloud storage cp "gs://cloud-samples-data/generative-ai/image/living-room.png" .
initial_image = Image.from_file(location="living-room.png")
mask = PIL_Image.new("L", initial_image._pil_image.size, 0)

target_size_w = int(2500 * eval("3/4"))
target_size = (target_size_w, 2500)
image_pil_outpaint, mask_pil_outpaint = pad_image_and_mask(
    initial_image._pil_image,
    mask,
    target_size,
    0,
    0,
)
image_pil_outpaint_image = Image(image_bytes=get_bytes_from_pil(image_pil_outpaint))
mask_pil_outpaint_image = Image(image_bytes=get_bytes_from_pil(mask_pil_outpaint))

raw_ref_image = RawReferenceImage(
    reference_image=image_pil_outpaint_image, reference_id=0
)
mask_ref_image = MaskReferenceImage(
    reference_id=1,
    reference_image=mask_pil_outpaint_image,
    config=MaskReferenceConfig(
        mask_mode="",  # TODO: Añade la mascara correspondiente a USER_PROVIDED
        mask_dilation=0.03,
    ),
)

edited_image = client.models.edit_image(
    model="",  # TODO: Añade el modelo de generación de imágenes
    prompt="",  # TODO: Añade el prompt
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_images(
    initial_image._pil_image, edited_image.generated_images[0].image._pil_image
)

### Ejercicio 6 - Outpainting - Extensión añadiendo contenido

Además de extender una imagen para llegar a nuevos formatos, podemos también personalizar la creación de contenido nuevo en el espacio generado por los modelos de generación de imágenes, modificando el parámetro `prompt`

In [ ]:
edited_image = client.models.edit_image(
    model=edit_model,
    prompt="Una lámpara de araña colgando del techo",
    reference_images=[raw_ref_image, mask_ref_image],
    config=EditImageConfig(
        edit_mode="",  # TODO: Añade el modo de edición correspondiente
        number_of_images=1,
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_images(
    initial_image._pil_image, edited_image.generated_images[0].image._pil_image
)